# El Data Model de Python (Parte 1): representación, igualdad y orden

Ya sabemos crear clases con datos y comportamiento. Pero hay algo que diferencia a una
clase "que funciona" de una clase que se siente **parte de Python**: su capacidad de
integrarse con las funciones y operadores nativos del lenguaje.

¿Por qué `len("hola")` funciona, o por qué puedo hacer `lista1 + lista2`, o
`sorted(numeros)`? Porque esos tipos implementan unos **métodos especiales** que
Python busca por debajo. A esos métodos, con doble guion bajo a cada lado
(`__init__`, `__repr__`...), se les llama ***dunders*** (de *double underscore*), y al
conjunto de reglas que los gobiernan se le llama el **Data Model** de Python.

En este vídeo aprenderemos a implementar los dunders de:

- **Representación**: `__repr__` y `__str__`.
- **Igualdad**: `__eq__`.
- **Orden**: `__lt__`, para poder usar `sorted()`.

Usaremos como ejemplo una clase `Libro` durante todo el vídeo.

> 📚 Documentación oficial de referencia:
> **[Data Model](https://docs.python.org/3/reference/datamodel.html)**.

---

## 1. El punto de partida: una clase "muda"

Empecemos con una clase `Libro` normal, sin ningún dunder más allá de `__init__`, y
veamos cómo se comporta con las funciones nativas:

In [ ]:
class Libro:
    def __init__(self, titulo, anio):
        self.titulo = titulo
        self.anio = anio


a = Libro("La soledad del corredor de fondo", 1959)
b = Libro("La soledad del corredor de fondo", 1959)

print(a)        # <__main__.Libro object at 0x...>  -> ilegible
print(a == b)   # False  -> mismos datos, pero se consideran distintos

Dos problemas que ya nos resultan familiares:

- Al imprimir, vemos la dirección de memoria, no el contenido.
- Dos libros con el mismo título y año se consideran **distintos**.

Python no sabe **qué significa** representar o comparar *nuestros* objetos. Tenemos que
enseñárselo, y eso se hace implementando los dunders adecuados.

---

## 2. Representación: `__repr__` y `__str__`

Hay **dos** formas de representar un objeto como texto, y conviene distinguirlas:

- `__repr__`: representación **para desarrolladores**. Pensada para depurar; idealmente
  debería ser inequívoca y, si se puede, parecerse a cómo se crearía el objeto. Es la
  que aparece en la consola y dentro de listas.
- `__str__`: representación **para el usuario final**. Más "bonita" y legible. Es la que
  usa `print()` y `str()`.

Si solo defines uno, define `__repr__`: Python lo usará también como respaldo cuando no
haya `__str__`.

In [ ]:
class Libro:
    def __init__(self, titulo, anio):
        self.titulo = titulo
        self.anio = anio

    def __repr__(self):
        return f"Libro({self.titulo!r}, {self.anio})"  # para desarrollar/depurar

    def __str__(self):
        return f"{self.titulo} ({self.anio})"            # para el usuario


libro = Libro("La soledad del corredor de fondo", 1959)
print(libro)     # La soledad del corredor de fondo (1959)            -> usa __str__
print([libro])   # [Libro('La soledad del corredor de fondo', 1959)]  -> dentro de una lista usa __repr__
libro            # en la consola de Jupyter también se ve el __repr__

> 💡 El `!r` dentro del f-string (`{self.titulo!r}`) aplica `repr()` al valor, por eso
> el título sale entre comillas: `Libro('La soledad del corredor de fondo', 1959)`. Así el `__repr__` se parece a
> la llamada que crearía el objeto, que es justo lo recomendable.

---

## 3. Igualdad: `__eq__`

Por defecto, `==` compara objetos por **identidad** (si son literalmente el mismo
objeto en memoria). Casi nunca es lo que queremos: normalmente dos libros con el mismo
título y año deberían considerarse **iguales**. Eso lo definimos con `__eq__`:

In [ ]:
class Libro:
    def __init__(self, titulo, anio):
        self.titulo = titulo
        self.anio = anio

    def __repr__(self):
        return f"Libro({self.titulo!r}, {self.anio})"

    def __eq__(self, otro):
        # comparamos por contenido: una tupla con los datos relevantes
        return (self.titulo, self.anio) == (otro.titulo, otro.anio)


a = Libro("La soledad del corredor de fondo", 1959)
b = Libro("La soledad del corredor de fondo", 1959)
c = Libro("Dune", 1965)

print(a == b)   # True   -> mismos datos
print(a == c)   # False  -> datos distintos

Un truco muy cómodo: comparar una **tupla** con los atributos relevantes contra la
tupla del otro objeto. Python ya sabe comparar tuplas elemento a elemento, así que nos
ahorramos escribir comparación por comparación.

---

## 4. Orden: `__lt__` para poder ordenar

Si intentamos ordenar una lista de libros, Python no sabe cuál va "antes":

In [ ]:
libros = [Libro("Dune", 1965), Libro("La soledad del corredor de fondo", 1959)]
try:
    sorted(libros)
except TypeError as e:
    print("Error:", e)

Para ordenar, `sorted()` necesita saber cuándo un objeto es "**menor que**" otro. Eso
lo define el dunder `__lt__` (*less than*). Decidamos, por ejemplo, ordenar por año:

In [ ]:
class Libro:
    def __init__(self, titulo, anio):
        self.titulo = titulo
        self.anio = anio

    def __repr__(self):
        return f"Libro({self.titulo!r}, {self.anio})"

    def __lt__(self, otro):
        return self.anio < otro.anio  # ordenamos por año


libros = [Libro("Dune", 1965), Libro("La soledad del corredor de fondo", 1959), Libro("Solaris", 1961)]
for libro in sorted(libros):
    print(libro)

Con solo `__lt__`, `sorted()` (y también `min()` y `max()`) ya saben trabajar con
nuestros libros.

> 💡 Si quieres todas las comparaciones (`<`, `<=`, `>`, `>=`) sin escribirlas una a
> una, el decorador `functools.total_ordering` las **genera automáticamente** a partir
> de `__eq__` y `__lt__`. Un decorador más que te ahorra trabajo, como los del vídeo
> anterior.

In [ ]:
from functools import total_ordering


@total_ordering
class Libro:
    def __init__(self, titulo, anio):
        self.titulo = titulo
        self.anio = anio

    def __repr__(self):
        return f"Libro({self.titulo!r}, {self.anio})"

    def __eq__(self, otro):
        return self.anio == otro.anio

    def __lt__(self, otro):
        return self.anio < otro.anio


a = Libro("La soledad del corredor de fondo", 1959)
b = Libro("Dune", 1965)

# total_ordering ha generado >, >=, <= a partir de __eq__ y __lt__
print(a < b)    # True
print(a >= b)   # False
print(b > a)    # True

---

## 5. Resumen

En este vídeo hemos aprendido a hacer que nuestras clases se **integren con Python**
implementando *dunders* del **Data Model**:

1. `__repr__` da una representación **para desarrolladores** (depuración, listas,
   consola); `__str__`, una **para el usuario** (`print`). Si solo defines uno, que
   sea `__repr__`.
2. `__eq__` define la igualdad **por contenido**; el truco es comparar una **tupla**
   con los datos relevantes.
3. `__lt__` habilita `sorted()`, `min()` y `max()`; `functools.total_ordering` genera
   el resto de comparaciones por ti.

| Dunder | Lo habilita |
|--------|-------------|
| `__repr__` | Representación para desarrolladores (consola, listas, depuración) |
| `__str__` | Representación para el usuario (`print`, `str`) |
| `__eq__` | Comparación con `==` por contenido |
| `__lt__` | Ordenar con `sorted()`, `min()`, `max()` |